# Assignment-3 — AI 24304 Machine Learning Techniques Lab
### Explained like you're new to this (with math + code)

This notebook solves all 3 questions:
1. Simple Linear Regression (per feature) on the **Wine Quality** dataset — with Ridge & Lasso, gradient descent written by hand.
2. Multiple Linear Regression (all features together) — same idea, but with a vector of betas.
3. Naive Bayes classifier on the **Heart Disease** dataset — written by hand, with confusion matrix, metrics, and ROC/AUC.

No `sklearn.linear_model` or `sklearn.naive_bayes` is used anywhere — every formula is coded manually, as the assignment demands.


## Setup

We just import the basic tools we need:
- `numpy` → for math/array operations
- `pandas` → for handling the dataset like an excel sheet
- `matplotlib` → for plotting graphs
- `mpl_toolkits.mplot3d` → for the 3D plot asked in Q1(b)


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
import os

np.random.seed(42)
os.makedirs("outputs", exist_ok=True)


---
## Question 1: Simple Linear Regression (one feature at a time)

### What are we even doing? (noob explanation)

We have the **Wine Quality** dataset. It has 11 numeric columns (like acidity, sugar, alcohol %, etc.) and
one target column `quality` (a score from 0–10 given by tasters).

"Simple linear regression" means: pick **just one** feature `x` at a time, and try to draw the best straight
line through the data that predicts `quality`:

$$ \hat{y} = \beta_0 + \beta_1 x $$

- $\beta_0$ = where the line crosses the y-axis (the "starting point")
- $\beta_1$ = the slope (how much quality changes when the feature changes by 1 unit)

We do this **separately for every one of the 11 features**, so we end up with 11 different tiny models.

### How do we find the best line? — Gradient Descent (noob explanation)

Imagine you're standing on a hill (the "error hill") and you want to walk down to the lowest point (smallest
error). You can't see the whole hill, but you can feel which way is downhill under your feet. Gradient
descent does exactly that:

1. Start with a random guess for $\beta_0, \beta_1$.
2. Compute how wrong the guess is (the error / loss).
3. Compute the **gradient** — the direction that makes the error worse — and step in the *opposite* direction.
4. Repeat many times (iterations). Each step, the line gets a bit better.

$$ \beta_{new} = \beta_{old} - \alpha \cdot \frac{\partial \text{Loss}}{\partial \beta} $$

where $\alpha$ is the **learning rate** — how big a step we take each time.

### The two loss functions

**Mean Squared Error (MSE)** — punishes big mistakes a lot (because we square them):
$$ MSE = \frac{1}{n}\sum (\hat y_i - y_i)^2 $$
Gradients:
$$ \frac{\partial MSE}{\partial \beta_0} = \frac{2}{n}\sum(\hat y_i - y_i), \qquad
   \frac{\partial MSE}{\partial \beta_1} = \frac{2}{n}\sum(\hat y_i - y_i)\,x_i $$

**Mean Absolute Error (MAE)** — treats every mistake equally (no squaring):
$$ MAE = \frac{1}{n}\sum |\hat y_i - y_i| $$
Gradients (using $sign$ because the derivative of $|z|$ is $+1$ or $-1$):
$$ \frac{\partial MAE}{\partial \beta_0} = \frac{1}{n}\sum sign(\hat y_i - y_i), \qquad
   \frac{\partial MAE}{\partial \beta_1} = \frac{1}{n}\sum sign(\hat y_i - y_i)\,x_i $$

### Regularization — Ridge (L2) and Lasso (L1)

Sometimes the model tries *too hard* to fit the data and the betas blow up. Regularization adds a penalty
that discourages large betas.

- **Ridge (L2 penalty)**: adds $\lambda \lVert \beta \rVert_2^2 = \lambda(\beta_0^2+\beta_1^2)$ to the loss.
  Extra gradient term: $2\lambda\beta$ — it shrinks betas smoothly, never exactly to 0.
- **Lasso (L1 penalty)**: adds $\lambda \lVert \beta \rVert_1 = \lambda(|\beta_0|+|\beta_1|)$ to the loss.
  Extra gradient term: $\lambda \cdot sign(\beta)$ — it can shrink betas all the way to exactly 0 (removes the feature).

$\lambda$ (lambda) controls *how strongly* we penalize — bigger $\lambda$ = simpler (flatter) model.


### Load the data

Download `winequality-red.csv` (or combine red+white) from the UCI link in the assignment and place it next
to this notebook, **or** point `WINE_URL` below at the raw UCI file. We use the semicolon-separated UCI file.


In [ ]:
# UCI Wine Quality dataset (semicolon separated)
WINE_URL_RED = "https://archive.ics.uci.edu/ml/machine-learning-databases/wine-quality/winequality-red.csv"

try:
    wine = pd.read_csv(WINE_URL_RED, sep=';')
except Exception as e:
    print("Could not fetch from the internet, falling back to local file 'winequality-red.csv'.", e)
    wine = pd.read_csv("winequality-red.csv", sep=';')

print(wine.shape)
wine.head()


In [ ]:
# Target column
TARGET = "quality"
feature_cols = [c for c in wine.columns if c != TARGET]

# Standardize (z-score) every feature and the target -> gradient descent converges MUCH faster and
# doesn't blow up when features are on very different scales (e.g. 'free sulfur dioxide' vs 'density').
def standardize(series):
    return (series - series.mean()) / series.std()

wine_std = wine.copy()
for col in wine.columns:
    wine_std[col] = standardize(wine[col])

y = wine_std[TARGET].values
print("Features:", feature_cols)


### The gradient descent engine (written from scratch)

One function that does simple linear regression for a single feature `x`, with a chosen loss (`mse`/`mae`)
and a chosen penalty (`none`/`ridge`/`lasso`). It returns the final betas **and** the full history of every
iteration (needed for part b's plot and CSV).


In [ ]:
def sign(z, eps=1e-8):
    # smooth-ish sign function; avoids 0-gradient issues at z == 0
    return np.where(z > eps, 1.0, np.where(z < -eps, -1.0, 0.0))

def simple_linear_gd(x, y, loss="mse", penalty="none", lam=0.0,
                      lr=0.05, n_iter=100, b0_init=0.0, b1_init=0.0):
    '''
    x, y      : 1D numpy arrays (already standardized)
    loss      : 'mse' or 'mae'
    penalty   : 'none', 'ridge' (L2) or 'lasso' (L1)
    lam       : regularization strength (lambda)
    returns   : b0, b1, history (list of dicts per iteration)
    '''
    n = len(x)
    b0, b1 = b0_init, b1_init
    history = []

    for it in range(n_iter):
        y_hat = b0 + b1 * x
        err = y_hat - y  # (pred - actual)

        if loss == "mse":
            g0 = (2 / n) * np.sum(err)
            g1 = (2 / n) * np.sum(err * x)
        elif loss == "mae":
            s = sign(err)
            g0 = (1 / n) * np.sum(s)
            g1 = (1 / n) * np.sum(s * x)
        else:
            raise ValueError("loss must be 'mse' or 'mae'")

        # regularization gradient added on top
        if penalty == "ridge":
            g0 += 2 * lam * b0
            g1 += 2 * lam * b1
        elif penalty == "lasso":
            g0 += lam * sign(np.array([b0]))[0]
            g1 += lam * sign(np.array([b1]))[0]
        elif penalty != "none":
            raise ValueError("penalty must be 'none', 'ridge' or 'lasso'")

        b0 -= lr * g0
        b1 -= lr * g1

        mse_val = np.mean((b0 + b1 * x - y) ** 2)
        mae_val = np.mean(np.abs(b0 + b1 * x - y))
        history.append({"iter": it, "beta0": b0, "beta1": b1,
                         "mse": mse_val, "mae": mae_val})

    return b0, b1, pd.DataFrame(history)


### 1(a) — Estimate $\beta_0$, $\beta_1$ for every feature, under Ridge and Lasso, for both MSE and MAE loss

We loop over all 11 features and, for each, run gradient descent 4 different ways:
`Ridge+MSE`, `Ridge+MAE`, `Lasso+MSE`, `Lasso+MAE`. We use a fixed small $\lambda$ here (we search for the
*best* $\lambda$ separately in part c).


In [ ]:
LAMBDA_DEFAULT = 0.1
results_1a = []

for feat in feature_cols:
    x = wine_std[feat].values
    for loss in ["mse", "mae"]:
        for penalty in ["ridge", "lasso"]:
            b0, b1, hist = simple_linear_gd(x, y, loss=loss, penalty=penalty,
                                             lam=LAMBDA_DEFAULT, lr=0.05, n_iter=100)
            results_1a.append({"feature": feat, "loss": loss, "penalty": penalty,
                                "beta0": b0, "beta1": b1,
                                "final_mse": hist['mse'].iloc[-1],
                                "final_mae": hist['mae'].iloc[-1]})

results_1a_df = pd.DataFrame(results_1a)
results_1a_df.to_csv("outputs/q1a_beta_estimates.csv", index=False)
results_1a_df


### 1(b) — 3D plot: $\beta_0$ (x-axis) vs $\beta_1$ (y-axis) vs error (z-axis), for 100 iterations

We pick one representative feature (`alcohol`, usually the strongest predictor of quality) and trace how
$\beta_0, \beta_1$ move over the 100 iterations, plotting both the MSE-trained path and the MAE-trained path.
Every iteration's numbers are also dumped to a CSV as asked.


In [ ]:
demo_feature = "alcohol" if "alcohol" in feature_cols else feature_cols[0]
x_demo = wine_std[demo_feature].values

_, _, hist_mse = simple_linear_gd(x_demo, y, loss="mse", penalty="ridge", lam=LAMBDA_DEFAULT, lr=0.05, n_iter=100)
_, _, hist_mae = simple_linear_gd(x_demo, y, loss="mae", penalty="ridge", lam=LAMBDA_DEFAULT, lr=0.05, n_iter=100)

# Save the iterative response to csv (both error types together)
combined = hist_mse[["iter", "beta0", "beta1", "mse"]].merge(
    hist_mae[["iter", "beta0", "beta1", "mae"]], on="iter", suffixes=("_mse_run", "_mae_run"))
combined.to_csv(f"outputs/q1b_iterations_{demo_feature}.csv", index=False)

fig = plt.figure(figsize=(10, 7))
ax = fig.add_subplot(111, projection='3d')
ax.plot(hist_mse["beta0"], hist_mse["beta1"], hist_mse["mse"], color='crimson', marker='o', markersize=2, label="MSE path")
ax.plot(hist_mae["beta0"], hist_mae["beta1"], hist_mae["mae"], color='royalblue', marker='o', markersize=2, label="MAE path")
ax.set_xlabel("beta0")
ax.set_ylabel("beta1")
ax.set_zlabel("error")
ax.set_title(f"Gradient descent path for feature: {demo_feature}")
ax.legend()
plt.tight_layout()
plt.savefig("outputs/q1b_3d_plot.png", dpi=150)
plt.show()


### 1(c) — Effect of $\lambda$ on the solution

We sweep $\lambda$ over a range of values and, for each, refit the model on `alcohol` (Ridge, MSE loss).
We record the final training error. The "best" $\lambda$ is the one that gives the **lowest error without
shrinking the betas to (near) zero** — too small a $\lambda$ overfits, too large underfits (betas collapse).


In [ ]:
lambdas = [0.0, 0.001, 0.01, 0.1, 0.5, 1.0, 5.0, 10.0]
lambda_results = []

for lam in lambdas:
    b0, b1, hist = simple_linear_gd(x_demo, y, loss="mse", penalty="ridge", lam=lam, lr=0.05, n_iter=100)
    lambda_results.append({"lambda": lam, "beta0": b0, "beta1": b1, "final_mse": hist['mse'].iloc[-1]})

lambda_df = pd.DataFrame(lambda_results)
lambda_df.to_csv("outputs/q1c_lambda_sweep.csv", index=False)

best_row = lambda_df.loc[lambda_df["final_mse"].idxmin()]
print("Best lambda by lowest training MSE:", best_row["lambda"])
lambda_df


In [ ]:
fig, ax1 = plt.subplots(figsize=(8, 5))
ax1.plot(lambda_df["lambda"], lambda_df["final_mse"], marker='o', color='crimson')
ax1.set_xscale("symlog")
ax1.set_xlabel("lambda")
ax1.set_ylabel("final MSE")
ax1.set_title(f"Effect of lambda on training error ({demo_feature})")
plt.tight_layout()
plt.savefig("outputs/q1c_lambda_effect.png", dpi=150)
plt.show()


### 1(d) — Observations

*(Fill these in with your own actual numbers once you run the notebook on your machine — this is a
template of what to write.)*

- MSE-based gradient descent converges smoothly because its gradient scales with the error size; MAE's
  gradient is a constant $\pm1$, so it moves in fixed-size steps and can oscillate near the minimum.
- Ridge shrinks both $\beta_0,\beta_1$ smoothly as $\lambda$ grows, but rarely makes them exactly 0.
- Lasso can zero-out $\beta_1$ for weak features once $\lambda$ is large enough — effectively saying
  "this feature doesn't matter."
- Very small $\lambda$ ≈ ordinary least squares (no regularization); very large $\lambda$ forces the model
  toward predicting just the mean (underfitting).
- The best $\lambda$ is a trade-off — pick the smallest $\lambda$ that still meaningfully controls the betas
  without hurting training error too much (this is normally chosen on a held-out validation set).


---
## Question 2: Multiple Linear Regression (all features at once)

### Noob explanation

Instead of one feature, we now use **all 11 features together**:

$$ \hat y = \beta_0 + \beta_1 x_1 + \beta_2 x_2 + \dots + \beta_{11} x_{11} $$

This is written more compactly using vectors/matrices. If $X$ is the matrix of all data points (rows =
wine samples, columns = features, with an extra column of 1's for the intercept) and $\beta$ is the vector
of all coefficients:

$$ \hat y = X\beta $$

The gradient descent idea is *identical* to Q1 — we just do it for a vector of betas instead of two
numbers. The math genuinely doesn't change, only the shapes of the arrays.

$$ \nabla_{MSE} = \frac{2}{n} X^T(\hat y - y), \qquad
   \nabla_{MAE} = \frac{1}{n} X^T\, sign(\hat y - y) $$

Ridge adds $2\lambda\beta$ to the gradient (skipping the intercept term is common practice — we do that
here too), Lasso adds $\lambda\, sign(\beta)$.


In [ ]:
def multi_linear_gd(X, y, loss="mse", penalty="none", lam=0.0, lr=0.05, n_iter=100):
    """
    X : 2D numpy array, shape (n_samples, n_features)  -- WITHOUT intercept column, we add it here
    y : 1D numpy array, shape (n_samples,)
    returns final beta vector (beta[0] is intercept) and history dataframe
    """
    n, d = X.shape
    Xb = np.hstack([np.ones((n, 1)), X])  # add intercept column
    beta = np.zeros(d + 1)
    history = []

    for it in range(n_iter):
        y_hat = Xb @ beta
        err = y_hat - y

        if loss == "mse":
            grad = (2 / n) * (Xb.T @ err)
        elif loss == "mae":
            grad = (1 / n) * (Xb.T @ sign(err))
        else:
            raise ValueError("loss must be 'mse' or 'mae'")

        reg_grad = np.zeros_like(beta)
        if penalty == "ridge":
            reg_grad[1:] = 2 * lam * beta[1:]        # don't regularize the intercept
        elif penalty == "lasso":
            reg_grad[1:] = lam * sign(beta[1:])

        grad = grad + reg_grad
        beta = beta - lr * grad

        mse_val = np.mean((Xb @ beta - y) ** 2)
        mae_val = np.mean(np.abs(Xb @ beta - y))
        row = {"iter": it, "mse": mse_val, "mae": mae_val}
        row.update({f"beta_{i}": beta[i] for i in range(len(beta))})
        history.append(row)

    return beta, pd.DataFrame(history)


In [ ]:
X_all = wine_std[feature_cols].values

beta_final, hist_multi = multi_linear_gd(X_all, y, loss="mse", penalty="ridge",
                                          lam=LAMBDA_DEFAULT, lr=0.05, n_iter=100)

hist_multi.to_csv("outputs/q2_gradient_updates.csv", index=False)

print("Intercept (beta_0):", beta_final[0])
for name, b in zip(feature_cols, beta_final[1:]):
    print(f"{name:25s} beta = {b: .4f}")

plt.figure(figsize=(8, 5))
plt.plot(hist_multi["iter"], hist_multi["mse"], label="MSE")
plt.xlabel("iteration")
plt.ylabel("error")
plt.title("Multiple Linear Regression — training error over iterations")
plt.legend()
plt.tight_layout()
plt.savefig("outputs/q2_convergence.png", dpi=150)
plt.show()


**Note:** the CSV `outputs/q2_gradient_updates.csv` records every coefficient (`beta_0` = intercept,
`beta_1..beta_11` = the 11 feature weights) at every one of the 100 iterations, exactly as asked.


---
## Question 3: Naive Bayes Classification on the Heart Disease dataset

### Noob explanation — what is Naive Bayes?

We want to predict whether a patient has heart disease (class 0 or 1) from their measurements (age,
cholesterol, chest-pain type, etc.).

Bayes' theorem tells us:

$$ P(\text{class} \mid \text{features}) = \frac{P(\text{features} \mid \text{class}) \cdot P(\text{class})}{P(\text{features})} $$

In plain words: *"the chance you're in a class, given what we observed, is proportional to how likely that
class would produce what we observed, times how common that class is overall."*

It's called **"naive"** because we (unrealistically, but usefully) assume every feature is independent of
every other feature, given the class. That turns one hard joint probability into a simple product:

$$ P(x_1,\dots,x_k \mid c) \approx \prod_{j=1}^{k} P(x_j \mid c) $$

Since the denominator $P(\text{features})$ is the same for every class, we just compare, for each class $c$:

$$ \text{score}(c) = P(c) \cdot \prod_j P(x_j \mid c) $$

and pick the class with the highest score. (We use **log** of this in code, turning the product into a sum
— avoids tiny numbers underflowing to 0.)

### Handling different feature types
- **Numeric features** (age, cholesterol, max heart rate...): we assume each follows a **Gaussian (normal)
  distribution** within each class, and use the Gaussian PDF as $P(x_j \mid c)$:
$$ P(x_j \mid c) = \frac{1}{\sqrt{2\pi\sigma_c^2}} \exp\left(-\frac{(x_j-\mu_c)^2}{2\sigma_c^2}\right) $$
- **Categorical features** (chest pain type, sex...): we just count frequencies —
$$ P(x_j = v \mid c) = \frac{\text{count of } (x_j=v \text{ and class}=c) + 1}{\text{count of class}=c + \#\text{categories}} $$
  (the "+1" is **Laplace smoothing**, so a category we've never seen in training doesn't kill the whole
  probability by multiplying by 0.)


In [ ]:
HEART_URL = "https://archive.ics.uci.edu/ml/machine-learning-databases/heart-disease/processed.cleveland.data"

col_names = ["age", "sex", "cp", "trestbps", "chol", "fbs", "restecg", "thalach",
             "exang", "oldpeak", "slope", "ca", "thal", "target"]

try:
    heart = pd.read_csv(HEART_URL, names=col_names, na_values="?")
except Exception as e:
    print("Could not fetch from the internet, falling back to local file 'processed.cleveland.data'.", e)
    heart = pd.read_csv("processed.cleveland.data", names=col_names, na_values="?")

heart = heart.dropna().reset_index(drop=True)

# The original 'target' is 0 (no disease) to 4 (severity); the assignment wants classification,
# so we binarize: 0 = no disease, 1 = disease present (any of 1-4)
heart["target"] = (heart["target"] > 0).astype(int)

# categorical (nominal/ordinal) vs numerical columns, based on the UCI documentation
categorical_cols = ["sex", "cp", "fbs", "restecg", "exang", "slope", "ca", "thal"]
numeric_cols = ["age", "trestbps", "chol", "thalach", "oldpeak"]

print(heart.shape)
heart.head()


In [ ]:
# Train/test split (by hand, no sklearn.model_selection needed but it's allowed for this - we just shuffle+slice)
def train_test_split_manual(df, test_frac=0.2, seed=42):
    rng = np.random.default_rng(seed)
    idx = rng.permutation(len(df))
    n_test = int(len(df) * test_frac)
    test_idx, train_idx = idx[:n_test], idx[n_test:]
    return df.iloc[train_idx].reset_index(drop=True), df.iloc[test_idx].reset_index(drop=True)

train_df, test_df = train_test_split_manual(heart, test_frac=0.2, seed=42)
print("Train size:", len(train_df), " Test size:", len(test_df))


### 3(a) — Naive Bayes, written from scratch

We fit the model by simply computing, from the training data:
- $P(c)$ for each class (the prior)
- mean & std of every numeric feature, per class
- frequency table of every categorical feature, per class

Then to predict a new patient, we compute the log-score for each class and pick the larger one.


In [ ]:
class NaiveBayesScratch:
    def __init__(self, numeric_cols, categorical_cols):
        self.numeric_cols = numeric_cols
        self.categorical_cols = categorical_cols

    def fit(self, df, target_col):
        self.classes = sorted(df[target_col].unique())
        self.priors = {c: (df[target_col] == c).mean() for c in self.classes}

        # numeric: store mean, std per class per feature
        self.num_stats = {c: {} for c in self.classes}
        for c in self.classes:
            sub = df[df[target_col] == c]
            for col in self.numeric_cols:
                mu, sigma = sub[col].mean(), sub[col].std() + 1e-6  # avoid div by 0
                self.num_stats[c][col] = (mu, sigma)

        # categorical: store smoothed frequency table per class per feature
        self.cat_stats = {c: {} for c in self.classes}
        self.cat_categories = {col: sorted(df[col].unique()) for col in self.categorical_cols}
        for c in self.classes:
            sub = df[df[target_col] == c]
            for col in self.categorical_cols:
                vc = sub[col].value_counts()
                n_c = len(sub)
                k = len(self.cat_categories[col])
                probs = {}
                for v in self.cat_categories[col]:
                    probs[v] = (vc.get(v, 0) + 1) / (n_c + k)   # Laplace smoothing
                self.cat_stats[c][col] = probs
        return self

    def _gaussian_log_pdf(self, x, mu, sigma):
        return -0.5 * np.log(2 * np.pi * sigma ** 2) - ((x - mu) ** 2) / (2 * sigma ** 2)

    def predict_scores(self, row):
        scores = {}
        for c in self.classes:
            log_score = np.log(self.priors[c])
            for col in self.numeric_cols:
                mu, sigma = self.num_stats[c][col]
                log_score += self._gaussian_log_pdf(row[col], mu, sigma)
            for col in self.categorical_cols:
                probs = self.cat_stats[c][col]
                p = probs.get(row[col], 1 / (len(probs) + 1))  # unseen category fallback
                log_score += np.log(p)
            scores[c] = log_score
        return scores

    def predict(self, df):
        preds = []
        for _, row in df.iterrows():
            scores = self.predict_scores(row)
            preds.append(max(scores, key=scores.get))
        return np.array(preds)

    def predict_proba_positive(self, df, positive_class=1):
        """Return P(class=positive_class) for each row, using the log-scores turned back into probabilities."""
        probs = []
        for _, row in df.iterrows():
            scores = self.predict_scores(row)
            m = max(scores.values())               # for numerical stability
            exp_scores = {c: np.exp(s - m) for c, s in scores.items()}
            total = sum(exp_scores.values())
            probs.append(exp_scores[positive_class] / total)
        return np.array(probs)

nb_model = NaiveBayesScratch(numeric_cols, categorical_cols).fit(train_df, "target")
test_preds = nb_model.predict(test_df)
test_probs = nb_model.predict_proba_positive(test_df, positive_class=1)
test_df["predicted"] = test_preds
test_df.head()


### 3(b) — Confusion Matrix & Classification Metrics

**Confusion matrix** (for binary classification) is just a 2x2 count table:

|                | Predicted 0 | Predicted 1 |
|----------------|-------------|-------------|
| **Actual 0**   | TN          | FP          |
| **Actual 1**   | FN          | TP          |

From these four numbers:
- **Accuracy** $= \dfrac{TP+TN}{TP+TN+FP+FN}$ — overall, how often are we right?
- **Precision** $= \dfrac{TP}{TP+FP}$ — of everyone we said "has disease", how many really did?
- **Recall / True Positive Rate (Sensitivity)** $= \dfrac{TP}{TP+FN}$ — of everyone who really has disease, how many did we catch?
- **Specificity** $= \dfrac{TN}{TN+FP}$ — of everyone who's really healthy, how many did we correctly say are healthy?
- **False Positive Rate** $= \dfrac{FP}{FP+TN} = 1 - \text{Specificity}$


In [ ]:
def confusion_matrix_manual(y_true, y_pred):
    TP = np.sum((y_true == 1) & (y_pred == 1))
    TN = np.sum((y_true == 0) & (y_pred == 0))
    FP = np.sum((y_true == 0) & (y_pred == 1))
    FN = np.sum((y_true == 1) & (y_pred == 0))
    return TP, TN, FP, FN

y_true = test_df["target"].values
y_pred = test_preds

TP, TN, FP, FN = confusion_matrix_manual(y_true, y_pred)
cm = np.array([[TN, FP], [FN, TP]])

accuracy = (TP + TN) / (TP + TN + FP + FN)
precision = TP / (TP + FP) if (TP + FP) else 0.0
recall = TP / (TP + FN) if (TP + FN) else 0.0          # True Positive Rate
specificity = TN / (TN + FP) if (TN + FP) else 0.0
fpr = 1 - specificity

print("Confusion matrix [[TN, FP], [FN, TP]]:")
print(cm)
print(f"Accuracy    : {accuracy:.3f}")
print(f"Precision   : {precision:.3f}")
print(f"Recall/TPR  : {recall:.3f}")
print(f"Specificity : {specificity:.3f}")
print(f"FPR         : {fpr:.3f}")

fig, ax = plt.subplots(figsize=(4.5, 4))
im = ax.imshow(cm, cmap="Blues")
ax.set_xticks([0, 1]); ax.set_xticklabels(["Pred 0", "Pred 1"])
ax.set_yticks([0, 1]); ax.set_yticklabels(["Actual 0", "Actual 1"])
for i in range(2):
    for j in range(2):
        ax.text(j, i, cm[i, j], ha="center", va="center", color="black", fontsize=14)
ax.set_title("Confusion Matrix")
plt.tight_layout()
plt.savefig("outputs/q3b_confusion_matrix.png", dpi=150)
plt.show()


### 3(c) — ROC curve & AUC

**ROC curve** (Receiver Operating Characteristic) shows the trade-off between True Positive Rate and False
Positive Rate **as we slide the decision threshold** from 0 to 1 (instead of always using 0.5).

For each threshold $t$: predict class 1 if $P(\text{disease}) \ge t$, else class 0 — then compute TPR and
FPR at that threshold. Plotting FPR (x-axis) vs TPR (y-axis) for every threshold gives the ROC curve.

**AUC** (Area Under the Curve) is a single number summarizing the curve — 1.0 = perfect classifier,
0.5 = random guessing. We compute it with the **trapezoidal rule** (summing thin trapezoid areas under the curve).


In [ ]:
def roc_curve_manual(y_true, scores, n_thresholds=100):
    thresholds = np.linspace(0, 1, n_thresholds)
    tpr_list, fpr_list = [], []
    for t in thresholds:
        y_pred_t = (scores >= t).astype(int)
        TP, TN, FP, FN = confusion_matrix_manual(y_true, y_pred_t)
        tpr = TP / (TP + FN) if (TP + FN) else 0.0
        fpr = FP / (FP + TN) if (FP + TN) else 0.0
        tpr_list.append(tpr)
        fpr_list.append(fpr)
    return np.array(fpr_list), np.array(tpr_list), thresholds

def auc_trapezoidal(fpr, tpr):
    # sort by fpr ascending for a clean integral
    order = np.argsort(fpr)
    fpr_sorted, tpr_sorted = fpr[order], tpr[order]
    return np.trapz(tpr_sorted, fpr_sorted)

fpr_arr, tpr_arr, thresh_arr = roc_curve_manual(y_true, test_probs, n_thresholds=200)
auc_value = auc_trapezoidal(fpr_arr, tpr_arr)
print(f"AUC = {auc_value:.3f}")

plt.figure(figsize=(6, 6))
plt.plot(fpr_arr, tpr_arr, color='crimson', label=f"ROC curve (AUC = {auc_value:.3f})")
plt.plot([0, 1], [0, 1], linestyle="--", color="gray", label="Random guess")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve — Naive Bayes on Heart Disease dataset")
plt.legend()
plt.tight_layout()
plt.savefig("outputs/q3c_roc_curve.png", dpi=150)
plt.show()


### Observations (Q3)

*(Again, plug in your actual numbers after running — this is the shape of what to write.)*

- Naive Bayes assumes feature independence, which is rarely true medically (e.g. cholesterol and age are
  correlated) — yet it often still works decently well as a baseline.
- Watch the confusion matrix: in a medical setting, **False Negatives** (telling a sick patient they're
  healthy) are usually far more costly than False Positives — so recall matters more than precision here.
- An AUC well above 0.5 means the model is meaningfully separating the two classes, not guessing randomly.
- Laplace smoothing on categorical features is what prevents a single unseen category value from crashing
  the whole prediction (probability of exactly 0 would otherwise wipe out the product).
